# PAN 2026 Proxy Evaluation

Reads directly from the exported dataset (`pan2026_proxy_dataset.zip`):
- `corpus.jsonl.gz` — source chunks (the pool to retrieve from)
- `queries.jsonl` — query chunks (plagiarized spans from suspicious docs)
- `qrels.txt` — ground truth relevance (TREC format)

Pipeline: e5-base-v2 → FAISS → Recall@1 at query chunk level

In [1]:
!pip install faiss-cpu sentence-transformers -q

from google.colab import drive
drive.mount('/content/drive')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 85.6 MB/s eta 0:00:00
Mounted at /content/drive


In [2]:
import shutil
import subprocess

print("Copying dataset zip from Drive...")
shutil.copy2(
    "/content/drive/MyDrive/pan2026_proxy_dataset.zip",
    "/content/pan2026_proxy_dataset.zip"
)
print("Unzipping...")
subprocess.run(["unzip", "-q", "/content/pan2026_proxy_dataset.zip", "-d", "/content/pan2026_proxy_dataset/"])
print("Done!")

Copying dataset zip from Drive...
Unzipping...
Done!


## Imports & Config

In [3]:
import json
import gzip
import numpy as np
import faiss
from pathlib import Path
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

In [10]:
DATASET_DIR = Path("/content/pan2026_proxy_dataset")
CORPUS_PATH  = DATASET_DIR / "corpus.jsonl.gz"
QUERIES_PATH = DATASET_DIR / "queries.jsonl"
QRELS_PATH   = DATASET_DIR / "qrels.txt"

# checkpoint directory on Drive (skip re-encoding if already done)
OUTPUT_DIR = Path("/content/drive/MyDrive/pan2025_proxy_retrieval_output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "intfloat/e5-base-v2"
BATCH_SIZE = 64
ENCODE_CHUNK_SIZE = 50_000

## Load Dataset

In [11]:
# Load corpus: list of {doc_id, default_text}
print("Loading corpus...")
corpus = []  # list of (doc_id, text)
with gzip.open(CORPUS_PATH, "rt", encoding="utf-8") as f:
    for line in f:
        obj = json.loads(line)
        corpus.append((obj["doc_id"], obj["default_text"]))
print(f"Corpus: {len(corpus)} source chunks")

# Load queries: list of {qid, query}
print("Loading queries...")
queries = []  # list of (qid, text)
with open(QUERIES_PATH, "r", encoding="utf-8") as f:
    for line in f:
        obj = json.loads(line)
        queries.append((obj["qid"], obj["query"]))
print(f"Queries: {len(queries)} query chunks")

# Load qrels: qid → set of relevant doc_ids
print("Loading qrels...")
qrels = {}  # qid → set[doc_id]
with open(QRELS_PATH, "r", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split()
        if len(parts) == 4:
            qid, _, doc_id, rel = parts
            if int(rel) > 0:
                qrels.setdefault(qid, set()).add(doc_id)
print(f"Qrels: {len(qrels)} queries have relevance judgements")
print(f"Avg relevant per query: {sum(len(v) for v in qrels.values()) / max(len(qrels),1):.1f}")

Loading corpus...
Corpus: 13705 source chunks
Loading queries...
Queries: 4676 query chunks
Loading qrels...
Qrels: 4627 queries have relevance judgements
Avg relevant per query: 1.7


## Encoding

In [12]:
def encode_texts(texts, model, prefix="passage: "):
    prefixed = [prefix + t for t in texts]
    return model.encode(
        prefixed,
        batch_size=BATCH_SIZE,
        show_progress_bar=True,
        normalize_embeddings=True,
        convert_to_numpy=True,
    ).astype(np.float32)


def encode_with_checkpoint(ids, texts, prefix, label, checkpoint_dir, chunk_size=ENCODE_CHUNK_SIZE):
    """
    Encode texts with checkpointing to Drive.
    If merged memmap already exists → load and return immediately.
    Returns: (memmap array [N, dim], list of N ids in same order)
    """
    mmap_path = checkpoint_dir / f"{label}_merged.dat"
    ids_path  = checkpoint_dir / f"{label}_ids.npy"
    dim_path  = checkpoint_dir / f"{label}_dim.npy"

    if mmap_path.exists() and ids_path.exists() and dim_path.exists():
        print(f"[{label}] Found existing embeddings — loading")
        saved_ids = list(np.load(ids_path, allow_pickle=True))
        dim = int(np.load(dim_path)[0])
        mmap = np.memmap(mmap_path, dtype="float32", mode="r", shape=(len(saved_ids), dim))
        print(f"[{label}] Loaded {len(saved_ids)} embeddings, dim={dim}")
        return mmap, saved_ids

    n = len(texts)
    all_saved_ids = []
    part_paths = []

    for start in range(0, n, chunk_size):
        end = min(start + chunk_size, n)
        part_idx  = start // chunk_size
        part_path = checkpoint_dir / f"{label}_part{part_idx:04d}.npz"
        part_paths.append(part_path)

        if part_path.exists():
            print(f"[{label}] Part {part_idx} already exists — skipping")
            part_data = np.load(part_path, allow_pickle=True)
            all_saved_ids.extend(list(part_data["chunk_ids"]))
        else:
            print(f"[{label}] Encoding {start}:{end} (part {part_idx})...")
            embs = encode_texts(texts[start:end], model, prefix=prefix)
            np.savez(part_path, embeddings=embs, chunk_ids=np.array(ids[start:end]))
            all_saved_ids.extend(ids[start:end])

    print(f"[{label}] Merging {len(part_paths)} part(s)...")
    first = np.load(part_paths[0], allow_pickle=True)
    dim = first["embeddings"].shape[1]
    del first

    merged = np.memmap(mmap_path, dtype="float32", mode="w+", shape=(n, dim))
    row = 0
    for part_path in part_paths:
        part_data = np.load(part_path, allow_pickle=True)
        embs = part_data["embeddings"]
        merged[row : row + len(embs)] = embs
        row += len(embs)
        del part_data, embs
    merged.flush()
    del merged

    np.save(ids_path, np.array(all_saved_ids))
    np.save(dim_path, np.array([dim]))
    print(f"[{label}] Done!")

    return np.memmap(mmap_path, dtype="float32", mode="r", shape=(n, dim)), all_saved_ids


print(f"Loading model: {MODEL_NAME}")
model = SentenceTransformer(MODEL_NAME)

corpus_ids   = [doc_id  for doc_id, _    in corpus]
corpus_texts = [text    for _,      text in corpus]
query_ids    = [qid     for qid,    _    in queries]
query_texts  = [text    for _,      text in queries]

print("\nEncoding corpus...")
corpus_embs, corpus_ids_ordered = encode_with_checkpoint(
    corpus_ids, corpus_texts, prefix="passage: ", label="src", checkpoint_dir=OUTPUT_DIR)

print("\nEncoding queries...")
query_embs, query_ids_ordered = encode_with_checkpoint(
    query_ids, query_texts, prefix="query: ", label="susp", checkpoint_dir=OUTPUT_DIR)

print("Done!")

Loading model: intfloat/e5-base-v2


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: intfloat/e5-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Encoding corpus...
[src] Part 0 already exists — skipping
[src] Merging 1 part(s)...
[src] Done!

Encoding queries...
[susp] Encoding 0:4676 (part 0)...


Batches:   0%|          | 0/74 [00:00<?, ?it/s]

[susp] Merging 1 part(s)...
[susp] Done!
Done!


## Build FAISS Index

In [13]:
print("Building FAISS index over corpus...")
dim = corpus_embs.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(np.array(corpus_embs))  # convert memmap to array for FAISS
print(f"Index built: {index.ntotal} vectors, dim={dim}")

Building FAISS index over corpus...
Index built: 13705 vectors, dim=768


## Retrieval & Evaluation

In [14]:
print("Retrieving top-1 for each query chunk...")

# Search: for each query chunk get top-1 source chunk
scores, indices = index.search(np.array(query_embs), 1)

tp = 0
fp = 0
fn = 0
results = []

for qid, score_row, idx_row in zip(query_ids_ordered, scores, indices):
    score        = float(score_row[0])
    retrieved_id = corpus_ids_ordered[int(idx_row[0])]
    relevant     = qrels.get(qid, set())

    hit = retrieved_id in relevant
    if hit:
        tp += 1
    else:
        fp += 1
        fn += 1

    results.append({
        "qid":       qid,
        "retrieved": retrieved_id,
        "score":     round(score, 4),
        "hit":       hit,
        "n_relevant": len(relevant),
    })

recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0

print("\n── Results ──────────────────────────────────────────")
print(f"  Recall@1 : {round(recall, 3)}")
print(f"  TP       : {tp}")
print(f"  FP       : {fp}")
print(f"  FN       : {fn}")
print(f"  Evaluated: {tp + fp} query chunks")
print("─────────────────────────────────────────────────────")

Retrieving top-1 for each query chunk...

── Results ──────────────────────────────────────────
  Recall@1 : 0.713
  TP       : 3332
  FP       : 1344
  FN       : 1344
  Evaluated: 4676 query chunks
─────────────────────────────────────────────────────
